In [ ]:
import cv2
import numpy as np
import urllib.request
from google.colab.patches import cv2_imshow


## [실습 1] RGB → YUV 컬러 모델 변환

**목표:** OpenCV를 사용하여 BGR 이미지를 YUV 컬러 공간으로 변환하고, 각 채널(Y, U, V)을 분리하여 시각화함으로써 컬러 채널의 특성을 이해합니다.

In [ ]:
# 1. 샘플 이미지 다운로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/smarties.png'
urllib.request.urlretrieve(url, 'smarties.png')

# 2. 이미지 로드 (BGR 형식)
img = cv2.imread('smarties.png')

# 3. BGR에서 YUV로 컬러 공간 변환
# Y: 밝기 정보, U/V: 색상 차이 정보
yuv_img = cv2.cvtColor(img, cv2.COLOR_BGR2YUV)

# 4. 채널 분리 (Y, U, V)
y_channel, u_channel, v_channel = cv2.split(yuv_img)

# 5. 시각화를 위해 채널 결합 (가로 방향)
combined_channels = np.hstack((y_channel, u_channel, v_channel))

# 6. 결과 출력
print("--- [실습 1 결과: 원본 이미지] ---")
cv2_imshow(img)

print("\n--- [실습 1 결과: YUV 채널 분리 (Y | U | V)] ---")
cv2_imshow(combined_channels)

## [실습 2] Gamma Correction (감마 보정)

**목표:** 룩업 테이블(LUT)을 사용하여 비선형적인 밝기 변환인 감마 보정을 구현하고, 감마 값에 따른 영상의 밝기 변화를 확인합니다.

In [ ]:
def adjust_gamma(image, gamma=1.0):
    # 1. 감마 값의 역수를 취함
    inv_gamma = 1.0 / gamma
    # 2. 0~255 범위에 대한 룩업 테이블(LUT) 생성
    # 픽셀값 / 255 를 감마 거듭제곱한 뒤 다시 255를 곱함
    table = np.array([((i / 255.0) ** inv_gamma) * 255
                      for i in np.arange(0, 256)]).astype("uint8")
    # 3. 생성된 테이블을 이미지에 적용
    return cv2.LUT(image, table)

# 샘플 이미지 다운로드 및 로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/fruits.jpg'
urllib.request.urlretrieve(url, 'fruits.jpg')
img = cv2.imread('fruits.jpg')

# 4. 다양한 감마 값 적용 (2.2: 밝게, 0.4: 어둡게)
bright_gamma = adjust_gamma(img, gamma=2.2)
dark_gamma = adjust_gamma(img, gamma=0.4)

# 5. 결과 가로 병합 및 출력
res = np.hstack((img, bright_gamma, dark_gamma))
print("--- [실습 2 결과: 원본 | 감마 2.2(Bright) | 감마 0.4(Dark)] ---")
cv2_imshow(res)

## [실습 3] 히스토그램 평활화 (Histogram Equalization)

**목표:** 영상의 명암 대비를 개선하기 위해 히스토그램 평활화를 수행합니다. 특히 컬러 영상의 경우 밝기(Y) 채널만 처리하여 색감 왜곡 없이 대비를 높이는 방법을 학습합니다.

In [ ]:
# 1. 샘플 영상 파일 다운로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/vtest.avi'
urllib.request.urlretrieve(url, 'vtest.avi')

# 2. 영상 로드 및 출력 설정
cap = cv2.VideoCapture('vtest.avi')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('vtest_equalized.mp4', fourcc, fps, (width, height))

# 3. 프레임별 처리 루프
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # 4. YUV 컬러 공간으로 변환 (밝기 정보만 다루기 위함)
    yuv = cv2.cvtColor(frame, cv2.COLOR_BGR2YUV)
    y, u, v = cv2.split(yuv)

    # 5. Y(밝기) 채널에만 히스토그램 평활화 적용
    y_equalized = cv2.equalizeHist(y)

    # 6. 처리된 Y 채널을 다시 병합하고 BGR로 복구
    yuv_equalized = cv2.merge([y_equalized, u, v])
    result_frame = cv2.cvtColor(yuv_equalized, cv2.COLOR_YUV2BGR)

    # 7. 처리된 프레임을 파일에 쓰기
    out.write(result_frame)

cap.release()
out.release()
print("--- [실습 3 결과] ---")
print("vtest_equalized.mp4 저장 완료")

## [실습 4] Convolution (Filtering) 기초

**목표:** 엠보싱, 샤프닝, 평균 블러 등의 커널을 직접 정의하고 `cv2.filter2D`를 사용하여 컨볼루션 연산을 수행함으로써 필터링의 원리를 이해합니다.

In [ ]:
# 샘플 이미지 다운로드
url = 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/baboon.jpg'
urllib.request.urlretrieve(url, 'baboon.jpg')
img = cv2.imread('baboon.jpg')

# 1. 다양한 필터 커널 정의
# (1) 엠보싱(Embossing): 인접 픽셀의 차이를 강조
emboss_kernel = np.array([[-1, -1, 0],
                          [-1,  0, 1],
                          [ 0,  1, 1]])

# (2) 샤프닝(Sharpening): 중심 픽셀을 강조하여 선명도 조절
sharp_kernel = np.array([[ 0, -1,  0],
                         [-1,  5, -1],
                         [ 0, -1,  0]])

# (3) 평균 (Average Blur): 5x5 크기로 주변 픽셀들의 평균값으로 부드럽게 처리
blur_kernel = np.ones((5, 5), np.float32) / 25

# 2. 컨볼루션 연산 수행 (cv2.filter2D)
emboss = cv2.filter2D(img, -1, emboss_kernel)
sharp = cv2.filter2D(img, -1, sharp_kernel)
blur = cv2.filter2D(img, -1, blur_kernel)

# 3. 결과 출력
print("--- [실습 4 결과: 1. 원본] ---")
cv2_imshow(img)

print("\n--- [실습 4 결과: 2. 엠보싱 필터] ---")
# 엠보싱은 128을 더해 회색조 배경에서 윤곽이 드러나게 함
cv2_imshow(cv2.add(emboss, 128))

print("\n--- [실습 4 결과: 3. 샤프닝 필터] ---")
cv2_imshow(sharp)

print("\n--- [실습 4 결과: 4. 평균 필터] ---")
cv2_imshow(blur)

## [실습 5] Gaussian Filtering (가우시안 블러)

**목표:** 가우시안 분포를 이용한 필터링을 통해 노이즈를 효과적으로 제거하면서 영상을 부드럽게 만드는 방법을 학습합니다.

In [ ]:
img = cv2.imread('baboon.jpg')

# 1. OpenCV 내장 가우시안 블러 함수 사용
# ksize가 커질수록 더 많이 흐려짐 (sigmaX는 0으로 설정 시 ksize에 따라 자동 계산)
blur_3x3 = cv2.GaussianBlur(img, (3, 3), 0)
blur_9x9 = cv2.GaussianBlur(img, (9, 9), 0)

# 2. 가우시안 커널 직접 생성하여 적용
# 5x5 가우시안 커널 생성 및 2차원 외적 연산
kernel_5x5 = cv2.getGaussianKernel(5, 0)
gaussian_kernel = np.outer(kernel_5x5, kernel_5x5)
custom_blur = cv2.filter2D(img, -1, gaussian_kernel)

# 3. 결과 출력
print("--- [실습 5 결과: 1. 원본] ---")
cv2_imshow(img)

print("\n--- [실습 5 결과: 2. 3x3 가우시안] ---")
cv2_imshow(blur_3x3)

print("\n--- [실습 5 결과: 3. 9x9 가우시안] ---")
cv2_imshow(blur_9x9)

print("\n--- [실습 5 결과: 4. 직접 정의한 5x5 커널] ---")
cv2_imshow(custom_blur)

## [실습 6] Sobel Edge Detection (소벨 에지 검출)

**목표:** 소벨 필터를 사용하여 영상의 미분값을 계산하고, 이를 통해 수직 및 수평 방향의 경계선(Edge)을 검출합니다.

In [ ]:
img = cv2.imread('baboon.jpg')
# 1. 처리를 위해 그레이스케일로 변환
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. 소벨 연산 수행
# x방향(수직선), y방향(수평선) 미분값 계산
sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

# 3. 미분값의 크기를 시각화하기 위해 절댓값 취함 및 8비트 변환
abs_sobel_x = cv2.convertScaleAbs(sobel_x)
abs_sobel_y = cv2.convertScaleAbs(sobel_y)

# 4. 두 방향의 에지 성분을 결합
sobel_combined = cv2.addWeighted(abs_sobel_x, 0.5, abs_sobel_y, 0.5, 0)

# 5. 결과 출력
print("--- [실습 6 결과: 원본 이미지] ---")
cv2_imshow(img)

print("\n--- [실습 6 결과: Sobel X (수직 경계)] ---")
cv2_imshow(abs_sobel_x)

print("\n--- [실습 6 결과: Sobel Y (수평 경계)] ---")
cv2_imshow(abs_sobel_y)

print("\n--- [실습 6 결과: 최종 결합된 에지] ---")
cv2_imshow(sobel_combined)

## [실습 7] Canny Edge Detection (캐니 에지 검출)

**목표:** 노이즈에 강하면서 정교한 에지 검출이 가능한 캐니 알고리즘의 동작 과정을 이해하고 이를 실습합니다.

In [ ]:
img = cv2.imread('fruits.jpg')

# 1. 전처리: 그레이스케일 변환 및 가우시안 블러를 통한 노이즈 제거
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
blur = cv2.GaussianBlur(gray, (5, 5), 0)

# 2. 캐니 에지 알고리즘 적용
# 두 임계값(Threshold)을 사용하여 강한 에지와 약한 에지를 구분 및 연결
edges = cv2.Canny(blur, 100, 200)

# 3. 결과 출력
print("--- [실습 7 결과: 원본 이미지] ---")
cv2_imshow(img)

print("\n--- [실습 7 결과: 캐니 에지 검출 결과] ---")
cv2_imshow(edges)

## [실습 8] Morphology (모폴로지 연산)

**목표:** 이진 영상에 침식, 팽창, 열림, 닫힘 연산을 적용하여 영상 내 객체의 형태를 다듬거나 노이즈를 제거하는 방법을 학습합니다.

In [ ]:
img = cv2.imread('smarties.png')

# 1. 이진화(Binarization) 수행
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
_, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

# 2. 모폴로지 연산을 위한 구조 요소(Kernel) 설정 (5x5 크기)
kernel = np.ones((5, 5), np.uint8)

# 3. 각 모폴로지 연산 수행
erosion = cv2.erode(binary, kernel, iterations=1)      # 침식 (작은 노이즈 제거)
dilation = cv2.dilate(binary, kernel, iterations=1)    # 팽창 (구멍 채우기)
opening = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)   # 열림 (침식 후 팽창)
closing = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)  # 닫힘 (팽창 후 침식)

# 4. 결과 출력 및 비교 (가로로 병합하여 출력)
res1 = np.hstack((binary, erosion, dilation))
res2 = np.hstack((binary, opening, closing))

print("--- [실습 8 결과: 상단: 이진화 원본 | 침식 | 팽창] ---")
cv2_imshow(res1)

print("\n--- [실습 8 결과: 하단: 이진화 원본 | 열림 | 닫힘] ---")
cv2_imshow(res2)

## [실습 9] Interpolation (보간법) 비교

**목표:** 영상 확대/축소 시 발생하는 픽셀값 결정을 위해 다양한 보간법(최근접 이웃, 양선형, 쌍입방)을 적용하고 시각적 차이를 분석합니다.

In [ ]:
img = cv2.imread('messi.jpg')

# 1. 테스트를 위해 이미지를 고의로 축소 (100x100)
small_img = cv2.resize(img, (100, 100), interpolation=cv2.INTER_AREA)

# 2. 축소된 이미지를 다시 5배 확대(500x500)하여 보간법 차이 확인
# (1) Nearest Neighbor: 가장 가까운 픽셀값 사용 (계단 현상 발생 가능)
nearest = cv2.resize(small_img, (500, 500), interpolation=cv2.INTER_NEAREST)

# (2) Bilinear: 인접 4개 픽셀의 가중치 평균 사용
linear = cv2.resize(small_img, (500, 500), interpolation=cv2.INTER_LINEAR)

# (3) Bicubic: 주변 16개 픽셀의 가중치 평균 사용 (고품질 확대)
cubic = cv2.resize(small_img, (500, 500), interpolation=cv2.INTER_CUBIC)

# 3. 결과 출력
print("--- [실습 9 결과: 1. 축소된 원본 (100x100)] ---")
cv2_imshow(small_img)

print("\n--- [실습 9 결과: 2. Nearest neighbor 확대] ---")
cv2_imshow(nearest)

print("\n--- [실습 9 결과: 3. Bilinear 확대] ---")
cv2_imshow(linear)

print("\n--- [실습 9 결과: 4. Bicubic 확대] ---")
cv2_imshow(cubic)

## [실습 10] Geometric Transform (기하학적 변환)

**목표:** Affine 변환 행렬을 이용하여 영상의 이동(Translation) 및 회전(Rotation)을 수행하는 방법을 이해합니다.

In [ ]:
img = cv2.imread('sudoku.png')
rows, cols = img.shape[:2]

# 1. 이동 변환 (Translation) 행렬 구성 및 적용
# x방향 100, y방향 50 이동
M_translation = np.float32([[1, 0, 100], [0, 1, 50]])
dst_translation = cv2.warpAffine(img, M_translation, (cols, rows))

# 2. 회전 및 크기 변환 (Rotation + Scale) 행렬 구성 및 적용
# 중심좌표 기준 45도 회전, 0.7배 축소 수행
M_rotation = cv2.getRotationMatrix2D((cols/2, rows/2), 45, 0.7)
dst_rotation = cv2.warpAffine(img, M_rotation, (cols, rows))

# 3. 결과 출력
print("--- [실습 10 결과: 1. 원본] ---")
cv2_imshow(img)

print("--- [실습 10 결과: 2. 이동 변환 (Translation)] ---")
cv2_imshow(dst_translation)

print("\n--- [실습 10 결과: 3. 회전 + 크기 변환 (Rotation + Scale)] ---")
cv2_imshow(dst_rotation)

## [실습 11] SIFT 스케일 공간 생성 (DoG Pyramid)

**목표:** SIFT 알고리즘의 핵심 단계인 가우시안 옥타브와 DoG(Difference of Gaussian) 피라미드를 구성하여 특징점 추출을 위한 준비 과정을 이해합니다.

In [ ]:
# 1. 영상 로드 및 흑백 변환
img = cv2.imread('butterfly.jpg')
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. 스케일 공간 설정을 위한 파라미터 정의
num_octaves = 4 # 총 4개의 옥타브 생성
s = 3           # 하나의 옥타브 내의 인터벌 수 (SIFT 표준값)
sigma0 = 1.6
k = 2**(1.0/s)  # 스케일 배수

gaussian_pyr = []
dog_pyr = []

# 초기 가우시안 이미지 생성
base_img = cv2.GaussianBlur(gray, (0, 0), sigmaX=sigma0, sigmaY=sigma0)

# 3. 피라미드 생성 루프
for o in range(num_octaves):
    current_octave_gaussians = [base_img]
    current_octave_dogs = []

    # 하나의 옥타브 내에서 s+3개의 가우시안 영상을 생성
    for i in range(1, s+3):
        sigma = sigma0 * (k ** i)
        gaussian_img = cv2.GaussianBlur(base_img, (0, 0), sigmaX=sigma, sigmaY=sigma)
        current_octave_gaussians.append(gaussian_img)

        # 4. 인접 가우시안 영상의 차이(DoG) 계산 (overflow 방지를 위해 float32 사용)
        dog_img = cv2.subtract(current_octave_gaussians[i].astype(np.float32),
                               current_octave_gaussians[i-1].astype(np.float32))
        current_octave_dogs.append(dog_img)

    gaussian_pyr.append(current_octave_gaussians)
    dog_pyr.append(current_octave_dogs)

    # 다음 옥타브를 위해 크기를 절반으로 줄임
    base_img = cv2.resize(current_octave_gaussians[s], (0, 0), fx=0.5, fy=0.5, interpolation=cv2.INTER_NEAREST)

# 5. 결과 시각화
print(f"--- [실습 11 결과: SIFT DoG Pyramid (Octaves: {num_octaves}, Intervals: {s})] ---")
for o in range(num_octaves):
    viz_list = []
    for d_img in dog_pyr[o]:
        # 차이값 강조를 위해 정규화 수행
        norm_img = cv2.normalize(d_img, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        viz_list.append(norm_img)

    octave_viz = np.hstack(viz_list)
    print(f"\n[Octave {o}] (Size: {octave_viz.shape[1]}x{octave_viz.shape[0]}) 가로로 결합된 DoG 영상")
    cv2_imshow(octave_viz)

## [과제 2] Butterfly 기하 변환 (Advanced Geometric Transform)

**목표:** 동치 좌표계와 복합 변환 행렬을 직접 구성하여 Forward Mapping을 수행하고, 발생하는 빈 공간(Hole)을 Backward Mapping과 쌍선형 보간법을 통해 해결하는 고급 기하 변환 기법을 구현합니다.

In [ ]:
def advanced_geometric_transform(img, angle_deg, tx, ty):
    # 1. 영상 크기 및 변환 정보 초기 설정
    h, w = img.shape[:2]
    rad = np.radians(angle_deg)
    diag = int(np.sqrt(w**2 + h**2))
    out_size = diag + 200 # 여백을 포함한 결과 영상 크기

    # 결과 영상 및 채워진 영역 기록용 마스크 초기화
    out_img = np.zeros((out_size, out_size, 3), dtype=np.uint8)
    filled_mask = np.zeros((out_size, out_size), dtype=np.uint8)

    # 2. [단계 1] 동치좌표계 구성 및 변환 행렬 생성
    # T1: 원점으로 이동, R: 회전 변환, T2: 출력 중심으로 이동 및 사용자 정의 이동
    T1 = np.array([[1, 0, -w/2], [0, 1, -h/2], [0, 0, 1]])
    R = np.array([[np.cos(rad), -np.sin(rad), 0], [np.sin(rad), np.cos(rad), 0], [0, 0, 1]])
    T2 = np.array([[1, 0, out_size/2 + tx], [0, 1, out_size/2 + ty], [0, 0, 1]])

    # 최종 복합 변환 행렬 H 및 보간용 역행렬 H_inv 계산
    H = T2 @ R @ T1
    H_inv = np.linalg.inv(H)

    # 3. [단계 2] Forward Mapping 수행
    # 입력 영상의 모든 픽셀을 새로운 좌표로 매핑
    for y in range(h):
        for x in range(w):
            p = np.array([x, y, 1])
            p_new = H @ p
            nx, ny = int(p_new[0]), int(p_new[1])
            if 0 <= nx < out_size and 0 <= ny < out_size:
                out_img[ny, nx] = img[y, x]
                filled_mask[ny, nx] = 1 # 값이 채워진 위치 기록

    # 4. [단계 3] Backward Mapping으로 Hole 채우기
    # Forward mapping 중 발생한 빈 공간(Hole)을 역추적 및 보간으로 해결
    for y in range(out_size):
        for x in range(out_size):
            if filled_mask[y, x] == 0: # 비어있는 픽셀(Hole)인 경우
                p_dst = np.array([x, y, 1])
                p_src = H_inv @ p_dst
                sx, sy = p_src[0], p_src[1]

                # 매핑되는 입력 좌표가 유효 범위 내인 경우 쌍선형 보간(Bilinear) 수행
                if 0 <= sx < w-1 and 0 <= sy < h-1:
                    x0, y0 = int(sx), int(sy)
                    dx, dy = sx - x0, sy - y0
                    # 주변 4개 픽셀값을 이용한 가중치 합산
                    pixel = (1-dx)*(1-dy)*img[y0, x0] + \
                            dx*(1-dy)*img[y0, x0+1] + \
                            (1-dx)*dy*img[y0+1, x0] + \
                            dx*dy*img[y0+1, x0+1]
                    out_img[y, x] = pixel.astype(np.uint8)

    return out_img

# Butterfly 영상 로드 및 변환 수행
butterfly = cv2.imread('butterfly.jpg')
print("--- [과제 12 결과: Butterfly 기하 변환 (회전 + 이동 + 보간)] ---")
hw_result = advanced_geometric_transform(butterfly, 225, 80, 80)
cv2_imshow(hw_result)